# Notebook 06 — Genie Code and Custom Skills

**What you'll do:**
1. Understand the difference between **Genie SQL** and **Genie Code**
2. Install the workshop's **skill file** so Genie Code knows your domain
3. Use a **prompt** to ask Genie Code to create a Genie SQL space for you
4. Use a second prompt to generate a full manufacturing analysis notebook

**The idea:** You describe *what* you want (a Genie space with these tables and business rules).
The skill teaches Genie Code *how* to do it (API calls, best practices, schema details).
You never write API code yourself.

**Before you start:** Run notebooks **03** (spaces) and **04** (benchmarks).

**Compute:** Serverless.


## Load workshop config


In [ ]:
%run ./00_workshop_config


In [ ]:
import re
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
host = w.config.host.rstrip("/")

def genie_ui_room_url(space_id):
    m = re.search(r"adb-(\d+)\.", host)
    o = m.group(1) if m else ""
    q = f"?o={o}" if o else ""
    return f"{host}/genie/rooms/{space_id}{q}"

_cfg = spark.table(full_table("workshop_config")).toPandas().set_index("key")["value"].to_dict()
PRIMARY_SPACE_ID = _cfg.get("genie_space_id", "")
if not PRIMARY_SPACE_ID:
    raise RuntimeError("genie_space_id not found in workshop_config. Run notebook 03 first.")
print(f"Primary space: {genie_ui_room_url(PRIMARY_SPACE_ID)}")


## Genie SQL vs Genie Code

| | Genie SQL | Genie Code |
|---|---|---|
| **What it does** | Answers data questions with SQL | Generates executable notebooks |
| **Where it lives** | AI/BI Genie space (web UI) | Databricks Assistant (notebook sidebar) |
| **How to teach it** | Instructions + curated Q-to-SQL in space config | Skill files in `.assistant/skills/` |
| **Best for** | Ad-hoc data questions, dashboards | Building spaces, complex analysis, multi-step workflows |

**Key insight:** The skill gives Genie Code domain knowledge (what OEE means, how tables relate).
The prompt tells it what to build. Together, Genie Code can create an entire Genie SQL space for you.


## Review the skill file

The skill ships with this workshop in the `skill/` folder:

```
GM-Genie-Workshop/
  skill/
    manufacturing-analytics_genie/
      SKILL.md
  notebooks/
    06_genie_code_skills    <-- you are here
```

It contains two sections:

- **Part 1 — Manufacturing domain knowledge:** Metrics (OEE, FPY, defect rate, scrap rate), table structures, join paths, shift definitions, event types
- **Part 2 — Genie space best practices:** How to write instructions, curated examples, benchmarks, and the API details Genie Code needs to call the REST API correctly

The skill is **generic** — it teaches domain concepts and API patterns, not your specific catalog or schema.


In [ ]:
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
parent_dir = "/".join(notebook_path.split("/")[:-1])
skill_source = f"{parent_dir}/skill/manufacturing-analytics_genie/SKILL.md"
print(f"Skill file: {skill_source}")
print("Open it in the workspace file browser to review.")


## Copy the skill into `.assistant/skills/`

Genie Code discovers skills from your **`.assistant/skills/`** folder.

1. In the workspace file browser, navigate to the `skill/` folder (path above)
2. Right-click **`manufacturing-analytics_genie`** folder and **Copy**
3. Navigate to `/Workspace/Users/<your_email>/.assistant/skills/`
   (create the folders if they don't exist)
4. Paste

The `_genie` suffix tells Genie Code to load this skill automatically.


In [ ]:
import os
user_email = w.current_user.me().user_name
expected_path = f"/Workspace/Users/{user_email}/.assistant/skills/manufacturing-analytics_genie/SKILL.md"
if os.path.exists(expected_path):
    size = os.path.getsize(expected_path)
    print(f"Skill found: {expected_path}  ({size:,} bytes)")
else:
    print(f"Skill NOT found at: {expected_path}")
    print(f"Copy from: {skill_source}")


## Prompt 1: Ask Genie Code to create a Genie SQL space

This prompt describes *what* you want — a Genie space with your tables, business rules,
and curated examples. The skill teaches Genie Code the API details behind the scenes.

### How to use
1. Open a **new, empty notebook**
2. Click the **Assistant** (sparkle icon) in the sidebar
3. Paste the prompt below
4. Genie Code generates the cells
5. Run them — you get a fully configured Genie space


In [ ]:
prompt_create_space = f"""
Create a Genie SQL space called "{GENIE_SPACE_PREFIX} - Genie Code Built".

IMPORTANT: Create the space directly — do NOT generate a notebook with API code.
Use your built-in ability to create and configure a Genie space.

My tables are in {CATALOG}.{SCHEMA}:
- plants (plant_id, plant_name, state, city)
- production_lines (line_id, line_name, plant_id, product_type, status)
- production_events (event_id, production_line_id, operator_id, event_type, event_date, defect_code, unit_serial_vin)
- operators (operator_id, first_name, last_name, shift, plant_id)
- quality_metrics_daily (plant_id, production_line_id, date, units_produced, defects_found, scrap_count, downtime_minutes, oee_score, first_pass_yield)
- safety_incidents (incident_id, production_line_id, incident_date, severity, description, root_cause)
- equipment_feedback (production_line_id, operator_id, feedback_date, feedback_type, description)

The space should help manufacturing managers answer questions about:
- OEE performance (target >= 85%, stored as 0-1 in oee_score, multiply by 100 for %)
- Defect rates by line, plant, and shift (count defect_detected vs unit_produced events)
- Scrap rates from quality_metrics_daily (100 * scrap_count / units_produced)
- Downtime trends and which lines need maintenance attention
- Shift quality comparison (join production_events to operators by operator_id)
- Safety incident tracking by severity

Key business rules to include in the space instructions:
- production_lines.status is STATIC (Active or Maintenance) — never add date filters on it
- Scrap rate always means the ratio 100 * scrap_count / units_produced, never a year or date
- Best quality shift = shift with fewest defect_detected events (MIN aggregation)
- Use exact string match (= 'Michigan') not ILIKE for state and plant_name filters
- For integer counts use CAST(... AS BIGINT), for rates use CAST(... AS DOUBLE)
- quality_metrics_daily uses column name 'date' (not metric_date)
- production_events.production_line_id joins to production_lines.line_id (not line_id to line_id)

Include 3 sample starter questions and 8-10 curated Q-to-SQL examples covering:
- Simple counts and filters
- Multi-table joins (events to lines to plants)
- Ratio calculations (defect rate, scrap rate, OEE)
- Shift-based analysis (events joined to operators)
- Date-range filtering on quality_metrics_daily

If a space with this title already exists, update it instead of creating a duplicate.
"""

print("=" * 70)
print("COPY THIS PROMPT INTO GENIE CODE (Assistant sidebar)")
print("=" * 70)
print(prompt_create_space)
print("=" * 70)
print(f"\nPrompt length: {len(prompt_create_space):,} chars")


## Prompt 2: Generate a manufacturing analysis notebook

This second prompt asks Genie Code to build a complete analytics notebook
(OEE trends, defect rates, safety dashboard) using the same domain knowledge from the skill.

### How to use
1. Open another **new, empty notebook**
2. Click the **Assistant** in the sidebar
3. Paste the prompt below
4. Run the generated cells


In [ ]:
prompt_analysis = f"""
Create a manufacturing quality analysis notebook using the tables in {CATALOG}.{SCHEMA}.
Use the manufacturing-analytics_genie skill for domain context.

Generate these analysis sections:

1. Setup — set catalog/schema, show table row counts for all 7 tables
2. OEE Analysis — average OEE % by plant, which plants are above/below 85%, monthly trend
3. Defect Analysis — overall defect rate, top 10 lines by defect rate, monthly trend
4. Downtime — total downtime hours by plant for 2024, top 5 lines
5. Safety — incident count by severity, monthly trend
6. Shift Insights — which shift has the most defects in Dec 2024 (join production_events to operators)
7. Executive Summary — total units, defect rate %, average OEE %, safety incidents, best plant

Use SQL cells where possible. Join paths:
- production_events.production_line_id -> production_lines.line_id -> plants.plant_id
- quality_metrics_daily.production_line_id -> production_lines.line_id
- oee_score and first_pass_yield are 0-1 scale (multiply by 100 for %)
"""

print("=" * 70)
print("COPY THIS PROMPT INTO GENIE CODE (Assistant sidebar)")
print("=" * 70)
print(prompt_analysis)
print("=" * 70)
print(f"\nPrompt length: {len(prompt_analysis):,} chars")


## What you learned

| Concept | What you did |
|---------|-------------|
| **Skill** | Installed a reusable `.md` file with manufacturing domain knowledge |
| **Prompt 1** | Asked Genie Code to create a Genie SQL space — it handled the API calls |
| **Prompt 2** | Asked Genie Code to build a full analysis notebook |

**The pattern:**

```
Skill (domain knowledge)    +    Prompt (what you want)    =    Output
manufacturing-analytics.md       "Create a space for            Genie SQL space with
  - OEE, defect rate defs          these 7 tables with            instructions, samples,
  - Join paths, thresholds         these business rules"          curated examples
  - API best practices
```

**Next:** Notebook **07** — A/B test your configured vs blank spaces.
